# Risk definition for residual load

Implements [`.claude/specs/02-Risk-Definition.md`](../../.claude/specs/02-Risk-Definition.md).

`notebooks/01_eda/team-EDA.ipynb` closed with one explicit open question: the two extremes of
`residual_load` are not one phenomenon — they differ in season, hour, day type, trend and
mechanism — and *"we need to decide whether the risk flag should treat them as one target or
two"*. This notebook answers that question and turns the answer into a concrete, reproducible
labelling methodology.

**This is not an EDA notebook.** Every finding it leans on was already established in
`team-EDA.ipynb` and is restated here in one or two lines, never re-derived.

## What this notebook produces

- Three **threshold bases** for each direction, derived from `data/smard.csv` itself with no
  hardcoded MWh figure: a causal `rolling` trailing-window quantile, a physical `zero` crossing
  (low direction only), and a whole-record `static` quantile kept as a reference.
- **Definition 1** — a per-day risk flag with a reason (the triggering hour and its value), under
  two day rules: `any` and a 3-hour persistence rule.
- **Definition 2** — per-hour flags using those same thresholds, and a derived time range per
  flagged day-direction.
- Two exported artifacts, `data/risk_labels_daily.csv` and `data/risk_labels_hourly.csv`, which
  the modeling spec can load directly.

## Conventions

- `time_series` is the main dataframe, `SERIES` its measured columns, `DERIVED` the engineered
  ones — inherited unchanged from [`03.1-setup.md`](../../.claude/specs/03.1-setup.md).
- `YEARS` is computed at run time; no literal calendar year appears in code.
- **Units:** `MWh` for every level and threshold. The dataset is energy data for the whole grid,
  so an hourly reading is treated as energy, not power.
- **Durations, never row counts.** Every duration-based rule below is expressed as a duration
  ("at least 3 hours"), not as a number of observations ("at least 3 rows"), so a later switch to
  SMARD's 15-minute resolution would not silently change the rule's meaning.

---

## 1 Setup

Inherited verbatim from [`03.1-setup.md`](../../.claude/specs/03.1-setup.md), the spec behind
`team-EDA.ipynb`'s setup section: the data-directory resolver, `time_series`, `SERIES`, `DERIVED`,
`YEARS`, `period_mean` / `period_energy`, `style_timeseries`, `DAY_NAMES` and the season mapping.
Nothing here is re-derived — see that spec and `notebooks/01_eda/team-EDA.ipynb` §1 for the
reasoning behind each piece.

`data/` is **gitignored**, so `data/smard.csv` does not come with a clone. Regenerate it by
running [`notebooks/API-connection.ipynb`](../API-connection.ipynb) top to bottom.

The data directory is resolved by walking **upward** from the working directory rather than by a
fixed `"../../data/smard.csv"`, so the notebook runs unmodified whether the kernel starts in
`notebooks/03_risk_classification/` or at the repo root.

In [ ]:
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Walk up from the working directory to the first parent holding a `data/` folder
DATA_DIR = next(
    (p / "data" for p in (Path.cwd(), *Path.cwd().parents) if (p / "data").is_dir()),
    None,
)
if DATA_DIR is None:
    raise RuntimeError(
        f"no data/ directory found in {Path.cwd()} or any parent — start the kernel inside the "
        "repository, then re-run."
    )
DATA = DATA_DIR / "smard.csv"

if not DATA.exists():
    raise FileNotFoundError(
        f"{DATA} not found. data/ is gitignored, so the file is not in a fresh clone — "
        "regenerate it by running notebooks/API-connection.ipynb top to bottom."
    )

print(f"pandas {pd.__version__} · numpy {np.__version__}")
print(f"Data directory: {DATA}")

### 1.1 Helpers

Copied unchanged from `team-EDA.ipynb` §1.1. `_complete_periods` drops calendar periods the data
does not fully cover — a plain `.resample()` produces fake edge dips. `style_timeseries` requires
an explicit `ylabel`, so no plot can ship without stating its unit.

In [ ]:
def _complete_periods(index, freq):
    """The calendar periods of `freq` that `index` covers completely.

    A period counts only if it starts not earlier than the first observation and ends not later than the last observation's closing edge.
    Both `period_mean` and `period_energy` defer to this.
    """
    periods = index.to_period(freq).unique().sort_values()
    complete = (
        periods.start_time >= index.min()) & (
        periods.end_time <= index.max() + pd.Timedelta("1h")
    )
    return periods[complete]


def period_mean(series, freq):
    """Mean of `series` per calendar period (`"W"`, `"M"`, ...), indexed by period start.

    Periods that the data does not cover completely are dropped, so the edges of a plot are not
    partial-period artefacts (half weeks / months, starting or ending a week on thursday).
    A period is considered an artifact if the range is "not fully covered".
    """
    agg = series.groupby(series.index.to_period(freq)).mean()
    agg = agg.loc[_complete_periods(series.index, freq)]
    agg.index = agg.index.start_time
    return agg


def period_energy(series, freq, drop_incomplete=True):
    """Per-period aggregate of `series` in both project reporting units.

    Returns a DataFrame indexed by period start:

    ``mwh_per_day``
        period sum / calendar days in the period — the energy view (MWh/day).
    ``avg_mw``
        period sum / hours **actually present** — the level view (MW).
        Deliberately not ``mwh_per_day / 24``: a month containing the spring Daylight-Saving-Time switch holds 743 hours, not 744.
    ``hours``, ``days``
        the two denominators, exposed so a comparison table needs no second copy of this
        arithmetic.

    Incomplete periods are dropped by the same `_complete_periods` rule as `period_mean`.
    """
    grouped = series.groupby(series.index.to_period(freq))
    total, hours = grouped.sum(), grouped.size()
    periods = total.index

    if drop_incomplete:
        keep = _complete_periods(series.index, freq)
        total, hours, periods = total.loc[keep], hours.loc[keep], keep

    # Freq-generic: 7 for every week, 28-31 for months. `days_in_month` would be "M"-only.
    days = (periods.end_time.normalize() - periods.start_time).days + 1

    # .to_numpy() on every right-hand side: aligning a PeriodIndex-backed Series against a
    # DatetimeIndex-derived array silently yields all-NaN.
    return pd.DataFrame(
        {
            "mwh_per_day": total.to_numpy() / days,
            "avg_mw": total.to_numpy() / hours.to_numpy(),
            "hours": hours.to_numpy(),
            "days": np.asarray(days),
        },
        index=periods.start_time,
    )

In [ ]:
def style_timeseries(ax, title, ylabel):
    """Custom grid, no box, year ticks.

    `ylabel` is required: every plot must state whether it shows MWh, average MW or MWh/day.
    """
    ax.set_title(
        title,
        loc="center",
        fontsize=15,
        pad=12
    )
    ax.set_xlabel("")
    ax.set_ylabel(
        ylabel,
        color="grey"
    )
    ax.grid(
        axis="y",
        color="0.9",
        linewidth=0.8
    )
    ax.set_axisbelow(True)

    for side in ("top", "right"):
        ax.spines[side].set_visible(False)

    ax.tick_params(
        colors="black",
        length=0  # hide ticks of values
    )
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_minor_locator(mdates.MonthLocator((1, 4, 7, 10)))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.yaxis.set_major_formatter(lambda v, _: f"{v:,.0f}")

### 1.2 Colour configuration

The subset of `team-EDA.ipynb` §1.2 this notebook needs. `TAIL_COLOR` carries over unchanged, so
the two directions read the same here as they do in the EDA: cool blue for the low/oversupply
tail, hot red for the high/undersupply tail.

In [ ]:
COLORS = {
    "blue":  "#2C6EBA",
    "red":   "#B10F0F",
    "black": "#1C1C1C",
    "gray":  "#AEB8C5",
    "muted": "#707B8C",
}

# Residual-load extremes, wherever they are shown as a pair:
# cool for oversupply (low), hot for undersupply (high).
TAIL_COLOR = {
    "low": COLORS["blue"],
    "high": COLORS["red"],
}

# One line style per threshold basis, used on every threshold plot below.
BASIS_STYLE = {
    "rolling": {"linestyle": "-", "linewidth": 2.0},
    "static": {"linestyle": "--", "linewidth": 1.4},
    "zero": {"linestyle": ":", "linewidth": 1.4},
}

DIRECTION_LABEL = {"high": "High residual load", "low": "Low / negative residual load"}

print(f"{len(TAIL_COLOR)} directions, {len(BASIS_STYLE)} bases configured")

### 1.3 Load and prepare

Dtype assertion guards against the German Excel-CSV conversion silently leaving a column as text.
Everything after this cell uses `time_series`.

In [ ]:
# The CSV headers exactly as notebooks/API-connection.ipynb writes them.
COLUMNS = {
    "Wind Offshore": "wind_off",
    "Wind Onshore": "wind_on",
    "Solar": "solar",
    "Grid Load": "grid_load",
    "Residual Load": "residual_load",
    "Forecast Wind + Solar": "fc_gen_wind_solar",
    "Forecast Grid Load": "fc_grid_load",
    "Forecast Residual Load": "fc_residual_load",
}

raw = pd.read_csv(DATA, delimiter=";", encoding="utf-8-sig")

assert set(raw.columns) == {"timestamp"} | set(COLUMNS), (
    f"unexpected CSV header: {sorted(set(raw.columns) ^ ({'timestamp'} | set(COLUMNS)))}"
)

raw = raw.rename(columns=COLUMNS)
raw["timestamp"] = pd.to_datetime(raw["timestamp"], format="%Y-%m-%d %H:%M")

for col in COLUMNS.values():
    raw[col] = raw[col].str.replace(",", ".").astype(float)

time_series = raw.set_index("timestamp").sort_index()
del raw  # the flat frame does not outlive the loading cell

# Positive is_float_dtype test, not `!= object`: under pandas 3 an unconverted German-decimal
# column lands as StringDtype, and `!= object` would wave it straight through.
assert all(
    pd.api.types.is_float_dtype(time_series[c]) for c in COLUMNS.values()
), time_series.dtypes

# Snapshot taken before any other cell can touch the frame, so the closing self-check can prove
# nothing in between mutated it.
LOADED = {
    "rows": len(time_series),
    "start": time_series.index.min(),
    "end": time_series.index.max(),
}

print(f"shape           : {time_series.shape[0]:,} rows x {time_series.shape[1]} columns")
print(f"index           : {time_series.index.min()}  ->  {time_series.index.max()}")
print(
    f"index monotonic : {time_series.index.is_monotonic_increasing}, "
    f"unique: {time_series.index.is_unique}"
)
time_series.head(3)

### 1.4 Derived columns, `SERIES` / `DERIVED`, and `YEARS`

`YEARS` is computed from the loaded data and is the only permitted source of year information in
the rest of the notebook. `spans_gap` marks the row *following* a gap in the hourly index — the
record's only gaps are the spring Daylight-Saving-Time switches, where the local hour 02:00 does
not exist.

In [ ]:
SERIES = [
    "wind_off", "wind_on", "solar", "grid_load", "residual_load",
    "fc_gen_wind_solar", "fc_grid_load", "fc_residual_load",
]

DAY_NAMES = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

# Meteorological seasons, with December assigned to the FOLLOWING year's winter.
SEASON_OF_MONTH = {
    12: "winter", 1: "winter", 2: "winter",
    3: "spring", 4: "spring", 5: "spring",
    6: "summer", 7: "summer", 8: "summer",
    9: "autumn", 10: "autumn", 11: "autumn",
}
SEASON_ORDER = ["winter", "spring", "summer", "autumn"]

time_series["renewables"] = time_series[["wind_on", "wind_off", "solar"]].sum(axis=1)
time_series["year"] = time_series.index.year
time_series["month"] = time_series.index.month
time_series["hour"] = time_series.index.hour
time_series["dow"] = time_series.index.dayofweek
time_series["is_weekend"] = time_series.index.dayofweek >= 5
time_series["date"] = time_series.index.date
time_series["season"] = pd.Categorical(
    time_series.index.month.map(SEASON_OF_MONTH), categories=SEASON_ORDER, ordered=True
)
time_series["season_year"] = time_series.index.year + (time_series.index.month == 12)

# Outputs True on the row FOLLOWING a gap. The first row is False (NaT comparison), not NaN.
time_series["spans_gap"] = time_series.index.to_series().diff() > pd.Timedelta("1h")

DERIVED = [
    "renewables", "year", "month", "hour", "dow", "is_weekend",
    "date", "season", "season_year", "spans_gap",
]

assert list(time_series.columns) == SERIES + DERIVED, list(time_series.columns)

# Plain ints, not np.int32: they end up in titles, labels and dict keys all over the notebook.
YEARS = sorted(int(y) for y in time_series["year"].unique())

print(f"{len(SERIES)} data columns + {len(DERIVED)} derived = {time_series.shape[1]} columns")
print(f"YEARS = {YEARS}")

### 1.5 Initial self-check

Structural only, and deliberately free of any hardcoded row count or date bound: the record's
extent is expected to change. The closing self-check re-runs these invariants plus a comparison
against `LOADED`.

In [ ]:
assert all(pd.api.types.is_float_dtype(time_series[c]) for c in SERIES), time_series[SERIES].dtypes
assert time_series.index.is_monotonic_increasing, "index is not sorted"
assert time_series.index.is_unique, "index has duplicate timestamps"
assert list(time_series.columns) == SERIES + DERIVED, list(time_series.columns)
assert YEARS == sorted(int(y) for y in time_series["year"].unique())

print("setup self-check passed")
print(f"  {len(SERIES)} float series, index sorted and unique")
print(f"  columns == SERIES + DERIVED ({len(SERIES) + len(DERIVED)} columns)")
print(f"  {LOADED['rows']:,} rows, {LOADED['start']} -> {LOADED['end']}, years {YEARS}")

### 1.6 What this notebook inherits from `team-EDA.ipynb`

Three findings carry real weight here. They are **cited, not re-derived** — no new evidence is
gathered for them in this notebook.

- **§3.7 — the two tails have almost mirror-image calendar signatures.** The low/negative extreme
  is overwhelmingly recent, concentrated in high-solar months, midday hours and weekends (the
  weekly demand minimum lining up with the daily solar maximum); the high extreme is weekday-only,
  spread evenly across years, in winter months, peaking in the evening with a secondary morning
  peak. §3.7 reads the low extreme as *growing and seasonal* and the high extreme as *stable and
  structural*. **This is the direct justification for treating high and low as two independently
  thresholded directions rather than one signed scale.** §3.7 also selected its 1 % tail *by rank*
  and labelled it explicitly a descriptive slice, not a risk definition.
- **§6.3 — the distribution is close to symmetric with a mild negative skew, and 1.26 % of all
  hours are already negative** — a small share, but growing and highly clustered in summer. There
  is no second mode and no sharp cutoff, so every threshold question here is a judgement call
  rather than a boundary the data hands us.
- **§6.5 — the low tail is stretching downward while the high end barely moves.** P10 falls faster
  than the median year on year; P90 hardly moves at all. This notebook's basis comparison exists
  specifically to make that asymmetry visible in threshold terms.

§6.5's **matched calendar window** (1 January to the record's last complete date, applied
identically to every year, derived from the data and never hardcoded) is the only admissible way
to compare years in this notebook — see §2 on why.

### 1.7 The residual-load identity

Before thresholding `residual_load`, confirm what it actually is in this record:

$$\text{residual\_load} = \text{grid\_load} - (\text{wind\_off} + \text{wind\_on} + \text{solar})$$

This is not a data-quality check for its own sake. If the identity holds, then SMARD *defines*
residual load as load minus wind and solar only — which means the project's "wind + solar only"
scope simplification costs **nothing for this target**. Every other generation source (biomass,
coal, hydro, ...) is already outside residual load by construction, not by our choice.

In [ ]:
implied = time_series["grid_load"] - time_series[["wind_off", "wind_on", "solar"]].sum(axis=1)
residual_error = time_series["residual_load"] - implied

abs_err = residual_error.abs()
typical = time_series["residual_load"].abs().median()
exact = (residual_error == 0).mean()
within_1 = (abs_err <= 1).mean()
over_1 = abs_err > 1

print(f"hours checked            : {len(residual_error):,}")
print(f"identity holds exactly   : {exact:6.2%}")
print(f"holds to within 1 MWh    : {within_1:6.2%}")
print(f"max deviation            : {abs_err.max():,.2f} MWh "
      f"({abs_err.max() / typical:.4%} of a typical |residual_load| of {typical:,.0f} MWh)")

if over_1.any():
    print(f"hours deviating > 1 MWh  : {int(over_1.sum())}, confined to "
          f"{residual_error[over_1].index.min():%Y-%m-%d} .. "
          f"{residual_error[over_1].index.max():%Y-%m-%d}")

**The identity holds to within rounding.** It is satisfied exactly for the large majority of
hours and to within 1 MWh for essentially all of them; the handful of larger deviations are
confined to a single short patch of the record and peak at a fraction of a per-mille of a typical
residual-load value — SMARD-side rounding and revision artefacts, not a different definition. At
the scale this notebook thresholds on (tens of thousands of MWh) they are immaterial.

The consequence is the point of the check: SMARD **defines** residual load as grid load minus wind
and solar, so the project's "wind + solar only" scope simplification costs **nothing for this
target**. Biomass, coal, hydro and the rest are outside residual load by construction, not by our
choice.

---

## 2 Defining risk

**Risk, in this project, is a `residual_load` magnitude extreme enough that the TSOs plausibly
need intervention measures to keep the grid balanced** — redispatch, reserve activation,
cross-border exchange, or curtailment.

`residual_load` is what is left of national demand once wind and solar have been subtracted (§1.7).
It is the quantity the remaining dispatchable fleet, storage, imports and exports have to close.
When it is extreme in *either* direction, closing it stops being routine.

That definition splits into **two mechanisms, not one scale**:

| Direction | Mechanism | Plausible intervention |
|---|---|---|
| **High** residual load | Least renewable cover relative to demand — the *Dunkelflaute*-type winter case: high demand, little wind, no solar | Upward redispatch, reserve activation, imports |
| **Low / negative** residual load | Renewable oversupply — wind and solar alone approach or exceed national demand | Downward redispatch, curtailment, exports, negative prices |

### Why two directions and not one signed scale

Because `team-EDA.ipynb` §3.7 already established that **the two extremes have almost mirror-image
calendar signatures** — they are not two ends of one phenomenon:

- The **low/negative** extreme is overwhelmingly **recent**, concentrated in high-solar months, at
  **midday**, and on **weekends** — the weekly demand minimum lining up with the daily solar
  maximum. §3.7 reads it as *growing and seasonal*, a consequence of renewable build-out.
- The **high** extreme is **weekday-only**, **spread evenly across the years** of the record, in
  **winter** months, peaking in the **evening** with a secondary morning peak. §3.7 reads it as
  *stable and structural*.

Different season, different hour, different day type, different mechanism. A single signed
threshold on one scale would have to pretend these are the same event seen from two sides. They
are not, so each direction gets its own threshold, computed independently, throughout this
notebook.

This is the open question `team-EDA.ipynb` closed on — *"we need to decide whether the risk flag
should treat them as one target or two"* — and **this notebook answers: two.** Whether a *model*
should then be trained as one two-sided target or two separate ones is a modelling decision, left
open in §7.

### 2.1 What this data cannot establish

These are **limits on the claim**, not caveats to bury. Each one bounds what the exported label is
allowed to be described as.

**1 — No margin.** The high direction is a *relative* extreme: the highest-residual-load hours
*in this record*. It is **not** a demonstrated approach to a capacity limit. SMARD's region-`DE`
series carry no installed-capacity, plant-availability or cross-border-capacity figures, so
"tight reserve margins" is simply not a claim this notebook can make — we can say an hour is
unusually high for this record, and nothing about how close the fleet came to running out.
(`team-EDA.ipynb` §3.7 uses the phrase "tight reserve margins" in passing; that reading goes
beyond what region-`DE` data supports, and is not adopted here.)

**2 — No regional detail.** The largest real driver of German redispatch is **north–south
transmission congestion** — high northern wind pushing against southern load — which can occur at
a perfectly moderate *national* residual load and is completely invisible in region-`DE` data. The
label produced here is therefore a **national-balance proxy**, and must be described as one
wherever it is presented. A day this label calls quiet may still have required intervention, and
the reverse.

**3 — No intervention record.** Nothing in `data/smard.csv` records whether a TSO actually
intervened on a given day. There is no ground truth here to validate against. The label is a
**plausibility proxy, never a validated outcome**.

### 2.2 What "risk" means here, precisely

This is a **level-based magnitude definition**, not a probability-of-intervention model. This
notebook produces the *candidate label* that a future model would be trained to predict — it does
not estimate any probability, and it fits no model of its own.

### 2.3 A trend claim this notebook does not make

**No text in this notebook asserts that the high tail is trending, in either direction.**
`team-EDA.ipynb` §3.7 reads it as spread across the years rather than trending, and nothing
computed here is entitled to overturn that. Two traps make it easy to believe otherwise:

- **The current year is partial.** The record ends mid-year, so comparing a complete early year
  against the in-progress final one is not a comparison at all. Any year-on-year view of
  `residual_load` would have to use §6.5's **matched calendar window** (1 January to the record's
  last complete date, applied identically to every year, with the end derived from the data rather
  than hardcoded).
- **Even matched, the window discards most of the high tail.** Roughly half of the high 1 % of
  hours falls in October–December, which a matched January-to-current window cuts out entirely.
  The year-to-year variation that survives is several times larger than any slope one could fit
  through eight noisy years.

**This notebook therefore shows no year-on-year view of the tails**, and the case against the
`static` basis in §3 rests on re-fetch instability and non-causality — arguments that do not
depend on any trend claim — rather than on how the tails move over time.

---

## 3 Threshold construction

Every threshold below is **derived from `data/smard.csv` at run time**. No MWh figure is hardcoded
anywhere in this notebook — the record's extent has already changed once (the fetch was widened
back to 2019) and will change again.

Three **bases** are computed, side by side, for comparison:

| Basis | Direction | Definition |
|---|---|---|
| `rolling` | both | Trailing 365-day quantile, using only days strictly **before** the day being labelled |
| `zero` | low only | The physical oversupply boundary, `0` |
| `static` | both | Whole-record quantile — kept as a **reference**, not recommended (§3.4) |

They are **not** presented as equals. §3.4 ranks them.

### 3.1 Rule constants, expressed as durations

Per the convention stated in the header: every duration-based rule is a **duration**, and the
number of observations it corresponds to is *derived* from the record's own resolution. At hourly
resolution "at least 3 hours" is 3 observations; if the record is ever switched to SMARD's
15-minute series, the same line becomes 12 observations with no edit.

The day-completeness rule works the same way — a day must carry at least 23/24 of its expected
observations. That accepts the 23-hour spring-DST days (of which this record has one per year) and
rejects any day left materially short by a future re-fetch.

In [ ]:
# Resolution is measured, not assumed.
RESOLUTION = time_series.index.to_series().diff().mode().iloc[0]

WINDOW = pd.Timedelta(days=365)      # trailing history for the rolling basis
PERSISTENCE = pd.Timedelta("3h")     # the `3h` day rule
DAY_COMPLETENESS = 23 / 24           # accepts the spring-DST day, rejects materially short days

MIN_RUN = int(PERSISTENCE / RESOLUTION)
EXPECTED_OBS_PER_DAY = int(pd.Timedelta("1D") / RESOLUTION)
MIN_OBS_PER_DAY = int(np.ceil(DAY_COMPLETENESS * EXPECTED_OBS_PER_DAY))

residual = time_series["residual_load"]

# One row per local calendar date — the same day boundary DERIVED["date"] uses, not a rolling 24h.
DAY = pd.Series(residual.index.normalize(), index=residual.index)
DAYS = pd.DatetimeIndex(sorted(DAY.unique()))
obs_per_day = residual.groupby(DAY).size().reindex(DAYS, fill_value=0)
day_complete = obs_per_day >= MIN_OBS_PER_DAY

print(f"resolution            : {RESOLUTION}  ->  {EXPECTED_OBS_PER_DAY} observations per full day")
print(f"persistence rule      : {PERSISTENCE}  ->  {MIN_RUN} consecutive observations")
print(f"day completeness      : >= {MIN_OBS_PER_DAY} of {EXPECTED_OBS_PER_DAY} observations")
print(f"calendar days         : {len(DAYS):,}  ({DAYS.min():%Y-%m-%d} .. {DAYS.max():%Y-%m-%d})")
print(f"observations per day  : {obs_per_day.value_counts().sort_index(ascending=False).to_dict()}")
print(f"days failing the completeness rule : {int((~day_complete).sum())}")

### 3.2 The rolling basis — the recommended one

For each calendar day `D`, the rolling thresholds are the quantiles of `residual_load` over the
trailing **365 days ending the day before `D`** — the window `[D-365, D-1]`.

**`D`'s own data never contributes to its own threshold.** That is what makes this basis *causal*:
the label for a day could have been produced on the morning of that day, which is the only
property that lets a day-ahead model be evaluated honestly against it.

A **full 365 days** of trailing history is required. Days with less get `NaN` — never a threshold
computed from a partial window. The tempting shortcut (allow a threshold once ~90 days exist, so
less of the record is wasted) produces a **seasonally biased** threshold, not merely a noisy one:
a window covering only winter yields a high threshold thousands of MWh above the full-year value,
which is then applied to spring and summer days. That bias is systematic and direction-dependent,
so it cannot be waved through as sampling error.

In [ ]:
def rolling_threshold(series, q, window=WINDOW):
    """Trailing-window quantile per calendar day, using only days strictly before it.

    The value for day D is the rolling quantile evaluated at the last observation of day D-1, so
    the window is exactly [D-365, D-1] and D itself is excluded. Days without a full `window` of
    trailing history are NaN.
    """
    roll = series.rolling(window).quantile(q)
    end_of_day = roll.groupby(series.index.normalize()).last()
    end_of_day.index = end_of_day.index + pd.Timedelta(days=1)  # end of D-1 applies to D
    thr = end_of_day.reindex(DAYS)
    return thr.where(thr.index >= series.index.min().normalize() + window)


def constant_threshold(value):
    """A threshold that is the same on every day (the `static` and `zero` bases)."""
    return pd.Series(float(value), index=DAYS)


def hourly_crossings(series, threshold_by_day, direction):
    """Per-observation crossing flags against each observation's own day's threshold.

    Returns nullable booleans: pd.NA wherever the day's threshold is undefined, so an undefined
    threshold can never masquerade as "not at risk".
    """
    thr = pd.Series(
        threshold_by_day.reindex(series.index.normalize()).to_numpy(), index=series.index
    )
    crossed = (series >= thr) if direction == "high" else (series <= thr)
    return crossed.astype("boolean").mask(thr.isna())


def day_rules(hourly_flag, threshold_by_day):
    """The `any` and `3h` day rules from per-observation crossing flags.

    `any`  — at least one observation crosses.
    `3h`   — the crossing persists for at least PERSISTENCE, consecutively, within the day.

    Consecutive *rows* count as consecutive: on the eight spring-DST days the missing 02:00 makes
    a run spanning it one hour longer in wall-clock terms than in rows. The approximation is noted
    rather than corrected, as 03.6 does for STL.

    Days with an undefined threshold, or materially short of a full day, are pd.NA in both rules
    rather than False.
    """
    filled = hourly_flag.fillna(False).astype(bool)
    any_rule = filled.groupby(DAY).any().reindex(DAYS, fill_value=False)

    run_id = (filled != filled.shift()).cumsum()
    run_len = filled.groupby([DAY, run_id]).transform("size").where(filled, 0)
    persist_rule = (run_len.groupby(DAY).max() >= MIN_RUN).reindex(DAYS, fill_value=False)

    valid = (threshold_by_day.notna() & day_complete).reindex(DAYS, fill_value=False)
    return any_rule.astype("boolean").mask(~valid), persist_rule.astype("boolean").mask(~valid)


_probe = rolling_threshold(residual, 0.99)
print(f"rolling basis defined for {int(_probe.notna().sum()):,} of {len(DAYS):,} days")
print(f"excluded for lack of a full {WINDOW.days}-day history : {int(_probe.isna().sum()):,} days "
      f"({DAYS.min():%Y-%m-%d} .. {_probe.first_valid_index() - pd.Timedelta(days=1):%Y-%m-%d})")

### 3.3 The zero and static bases

**`zero` (low direction only): `low_threshold_zero = 0`.** This is the one basis with a meaning
outside this dataset. Below zero, wind and solar alone exceed national demand, so the surplus must
be exported, curtailed, or paid away at negative prices. It takes no percentile parameter and does
not move when the record is re-fetched.

**The high direction has no equivalent physical anchor in this data.** There is no level of
residual load that region-`DE` data marks as "the fleet runs out here" — that would need
installed-capacity and availability figures SMARD does not carry in these series (§2.1). The
asymmetry between the two directions is therefore **deliberate, not an oversight**: the low
direction gets a physical boundary because one exists; the high direction does not, because none
is observable here.

**`static`: the whole-record quantile**, computed once over the full currently-loaded record.

In [ ]:
P_LEVELS = [0.005, 0.01, 0.02, 0.05]
P_DEFAULT = 0.01  # carried into Definitions 1 and 2 — see §3.5

# Which bases exist per direction. The low direction has one more, because zero only means
# something on that side.
BASES = {"high": ["rolling", "static"], "low": ["rolling", "zero", "static"]}


def build_thresholds(p):
    """Every threshold series, per direction x basis, at percentile level `p`."""
    return {
        ("high", "rolling"): rolling_threshold(residual, 1 - p),
        ("high", "static"): constant_threshold(residual.quantile(1 - p)),
        ("low", "rolling"): rolling_threshold(residual, p),
        ("low", "zero"): constant_threshold(0.0),
        ("low", "static"): constant_threshold(residual.quantile(p)),
    }


THRESHOLDS = build_thresholds(P_DEFAULT)

print(f"At the default level p = {P_DEFAULT:.1%}:")
for (direction, basis), thr in THRESHOLDS.items():
    if basis == "rolling":
        print(f"  {direction:4} {basis:7} : {thr.min():>10,.0f} .. {thr.max():>10,.0f} MWh "
              f"(mean {thr.mean():>9,.0f}, varies by day)")
    else:
        print(f"  {direction:4} {basis:7} : {thr.iloc[0]:>10,.0f} MWh (constant)")

### 3.4 Ranking the bases

Unlike the surrounding EDA specs, this notebook does **not** stay neutral between the three.

#### The static basis has two defects

**1 — It is unstable under re-fetch.** The static threshold is a property of *the record's extent*,
not of the grid. Widening the fetch back to 2019 — which this team has already done once — **raised**
the low threshold and retroactively un-flagged days that were flagged before; extending the end
date pushes it the other way. A label that silently changes when you download more data is not a
label.

**2 — It is not causal.** A day in 2022 is judged against quantiles computed from data including
2025–2026. It cannot be reproduced in a live day-ahead pipeline, because most of its inputs had
not happened yet.

#### The rolling basis has costs too

It is **undefined for the record's first 365 days**, and it **moves as history accumulates**, so
the same absolute MWh level can be "risky" in one period and ordinary in another — identical
physics, different label. That is the price of a stable base rate, and it is the trade this
ranking accepts.

#### The ranking

- **Recommended as label bases: `rolling` (both directions) and `zero` (low direction).**
  Rolling is the only causal basis and the only one that keeps the positive rate comparable across
  a chronological split (§3.6 measures this). Zero is the only one that means anything independent
  of this record.
- **Demoted: `static`.** It is still computed, still exported, and still drawn as a reference line
  on the threshold plots — so the comparison stays visible and the team can overrule this
  judgement — but it is **not recommended as a label basis**, for the two reasons above.

Both directions keep all their bases in the export as parallel, clearly labelled candidate
columns. This ranking is a recommendation for the modelling spec to act on, not a deletion of the
alternatives.

### 3.5 Sensitivity to the percentile level

One row per candidate level, per direction. The static threshold is shown as a reference; the
hour- and day-shares are computed on the **rolling** basis (the recommended one), over the days
where it is defined — the first 365 days are excluded from both numerator and denominator.

The gap between the hour-share and the day-share is the point of the table: **a day is flagged if
any single one of its hours crosses**, so the day-share runs several times the hour-share. The
`3h` column shows how much of that survives a persistence requirement.

In [ ]:
def sensitivity(direction):
    rows = []
    for p in P_LEVELS:
        q = 1 - p if direction == "high" else p
        roll = rolling_threshold(residual, q)
        hourly = hourly_crossings(residual, roll, direction)
        any_rule, persist_rule = day_rules(hourly, roll)
        rows.append({
            "level": f"{p:.1%}",
            "static (MWh)": f"{residual.quantile(q):,.0f}",
            "rolling mean (MWh)": f"{roll.mean():,.0f}",
            "rolling min (MWh)": f"{roll.min():,.0f}",
            "rolling max (MWh)": f"{roll.max():,.0f}",
            "hours flagged": f"{hourly.mean():.2%}",
            "days flagged (any)": f"{any_rule.mean():.2%}",
            "days flagged (3h)": f"{persist_rule.mean():.2%}",
        })
    return pd.DataFrame(rows).set_index("level")


for direction in ("high", "low"):
    print(f"\n{DIRECTION_LABEL[direction]} — sensitivity across candidate levels "
          f"(shares on the rolling basis)")
    display(sensitivity(direction))

**Findings.**

- **The day-share runs four to six times the hour-share** at every level, in both directions. That
  is the `any` rule doing what it says: one crossing hour flags the whole day. The multiple is
  largest at the tightest levels, where crossings are rarest and most isolated.
- **Requiring the crossing to persist for 3 hours removes between a third and nearly half of
  flagged days in the high direction, and consistently less in the low direction.** Low-direction
  events are the more persistent of the two, which fits §3.7's reading of them as a broad midday
  solar trough rather than a sharp evening peak.
- **The high direction's hour-share lands slightly *below* its nominal level** (~0.9 % of hours at
  a nominal 1 %). A threshold built from trailing history is applied to a period whose high tail is
  no more extreme than that history, so slightly fewer hours clear it than the quantile would
  suggest on its own data.
- **The low direction's hour-share lands well *above* nominal** — around 1.7 % of hours at a
  nominal 1 %, nearly double. This is not a bug: it is §6.5's downward stretch made visible.
  Because the low tail keeps falling, a threshold computed from the trailing year is systematically
  too lenient for the year that follows, and more hours breach it than the nominal rate. The low
  direction is chasing a moving target in a way the high direction is not.
- **The static and rolling-mean thresholds diverge sharply in the low direction and barely at all
  in the high direction.** In the high direction the two sit within a few hundred MWh of each
  other at every level. In the low direction the static threshold is far below the rolling mean,
  because the whole-record quantile is dragged down by the recent, much more negative years and
  then applied backwards across a record where those values did not occur.

**1 % is carried into Definitions 1 and 2**, for continuity with the 1 % tail-by-rank framing
`team-EDA.ipynb` §3.7 already used. This is a **recommendation for comparability, not a
re-derivation of that number** — and §3.7 itself called that slice descriptive rather than a risk
definition. Nothing here narrows the export: all four levels remain equally defensible, and the
other bases stay in the file.

### 3.6 Positive rate by year — and why a chronological split is not exchangeable

Share of days flagged per calendar year, at the default level, under the `any` rule.

In [ ]:
positive_rate = pd.DataFrame(
    {f"{d} · {b}": day_rules(hourly_crossings(residual, thr, d), thr)[0]
     for (d, b), thr in THRESHOLDS.items()}
)
by_year = positive_rate.groupby(positive_rate.index.year).mean()
by_year.index.name = "year"

days_per_year = positive_rate.groupby(positive_rate.index.year).size()
partial = days_per_year[days_per_year < 365]

display(
    by_year.style.format("{:.1%}", na_rep="—")
    .set_caption(f"Share of days flagged, `any` rule, p = {P_DEFAULT:.1%}")
)
print(f"days covered per year: {days_per_year.to_dict()}")
if len(partial):
    print(f"PARTIAL year(s): {partial.to_dict()} — rates for these are not comparable to full years")

**Findings — and a warning the modelling spec has to act on.**

**The low direction's base rate is strongly non-stationary.** Under the `static` and `zero` bases
it is **exactly empty for the record's first four years** and then climbs steeply, ending an order
of magnitude higher than it began. This is not a thresholding artefact: negative residual-load
hours simply did not occur in Germany before 2023, and their share has risen every year since. A
basis with a fixed level therefore does not measure the same thing in 2020 as in 2026.

**The rolling basis is markedly more stable** — the low direction's rate stays within a band a
few-fold wide instead of going from zero to a third of all days. It is *not* stationary either,
and should not be described as such, but it is the only basis under which the early and late
record are remotely comparable.

**The high direction's rate shows no trend under either basis** — it moves around within a band,
which is consistent with `team-EDA.ipynb` §3.7's reading. Nothing here is evidence of a trend in
the high tail, in either direction (§2.3).

**The final year is partial.** The record ends mid-year, so that row covers only part of a year
and is **not comparable to the full years above it**. Because the low phenomenon is summer-heavy
and the high phenomenon winter-heavy, a January-to-September slice *inflates* the low rate and
*deflates* the high one. Read the last row as a fragment, never as the end of a trend.

**The consequence.** A chronological train/test split has **non-exchangeable base rates** for the
low label: any classifier metric computed across it — precision, recall, F1, AUC — is comparing
periods where the label means different things, and is not comparable.

**Recommendation.** Keep the **full record** for training the underlying residual-load regression;
grid load is stable and that history is genuinely useful. But note that the low *label* only
becomes populated partway through the record under any fixed standard. The record is **not
narrowed here** — that decision belongs to the modelling spec, with this table in hand.

### 3.7 How the bases diverge over time

The rolling threshold as a time series per direction, with the static threshold as a horizontal
reference and — for the low direction — zero as a second reference, so the divergence is visible
directly rather than inferred from summary statistics.

The rolling line moves in **steps rather than smoothly**: an extreme quantile of an 8,760-hour
window only shifts when an extreme value enters or leaves that window, which happens on some days
and not others.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)

for ax, direction in zip(axes, ("high", "low")):
    roll = THRESHOLDS[(direction, "rolling")]
    ax.plot(roll.index, roll.to_numpy(), color=TAIL_COLOR[direction],
            **BASIS_STYLE["rolling"], label=f"rolling ({WINDOW.days}-day trailing)")
    ax.axhline(THRESHOLDS[(direction, "static")].iloc[0], color=COLORS["muted"],
               **BASIS_STYLE["static"], label="static (whole record)")
    if direction == "low":
        ax.axhline(0, color=COLORS["black"], **BASIS_STYLE["zero"],
                   label="zero (physical oversupply boundary)")
    style_timeseries(
        ax,
        f"{DIRECTION_LABEL[direction]} — threshold at p = {P_DEFAULT:.1%}",
        "Residual load threshold (MWh)",
    )
    ax.legend(frameon=False, loc="best")

fig.tight_layout()
plt.show()

for direction in ("high", "low"):
    roll = THRESHOLDS[(direction, "rolling")].dropna()
    print(f"{direction:4} rolling threshold drift: range {roll.max() - roll.min():>9,.0f} MWh "
          f"(min {roll.min():>9,.0f}, max {roll.max():>9,.0f}, std {roll.std():>8,.0f})")

**Findings.**

**The low threshold drifts several times more than the high threshold** — exactly as §6.5's
finding predicts, now in threshold terms rather than as an adjective. The high line wanders inside
a relatively narrow band with no sustained direction. The low line marches steadily downward
across the record and **crosses zero partway through**: for the early years the 1 % low threshold
sits well *above* zero, meaning the bottom 1 % of hours were still comfortably positive and the
`zero` basis flagged nothing at all; by the end of the record it sits well below zero.

That crossing is the single most important thing on this figure. It is why the `zero` and
`static` bases are empty for the early record and why their base rates in §3.6 look the way they
do — and it is the clearest visual argument for the ranking in §3.4. The high direction's static
line is a reasonable summary of its rolling line; the low direction's static line is a poor
summary of anything, sitting far below the rolling threshold for most of the record and far above
it at the end.